In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


#### Get subzone data

In [3]:

def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones = pd.read_csv(subzones_filepath)

    subzones = subzones.map(lambda s: s.lower() if isinstance(s, str) else s)
    return subzones

In [4]:
subzone_df = get_subzone_data(2014)
subzone_df.head(3)

,subzone_n,pln_area_n,beach_area,business_2,educational_institution,utility,special_use,waterbody,port_/_airport,park,...,reserve_site,white,place_of_worship,business_2_-_white,hotel,residential_/_institution,business_park_-_white,business_park,agriculture,pri_classification
0,ang mo kio town centre,ang mo kio,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.00000,...,3618.378339,0.0,3158.691818,0.0,0.0,0.0,0.0,0.0,0.0,residential
1,cheng san,ang mo kio,0.0,0.0,69353.966728,5449.975328,0.0,0.0,0.0,0.00000,...,66034.996250,0.0,2670.252332,0.0,0.0,0.0,0.0,0.0,0.0,residential
2,chong boon,ang mo kio,0.0,0.0,100775.355941,3683.124722,0.0,0.0,0.0,30024.94591,...,0.000000,0.0,3709.155602,0.0,0.0,0.0,0.0,0.0,0.0,residential


#### Resident Density

In [5]:
def get_resident_data(year):
    """
    Args
    ------
    year: int
        year of the resident data (resident numbers and age group) you want (2020, 2015, 2010)

    Returns
    ------
    Dataframe containing the subzone names, planning area names and resident numbers and age group.
    """
    residential_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/{year}_age_group_planning_area_subzone.xlsx")

    residential_data = pd.read_excel(residential_filepath, sheet_name = "subzone")
    residential_data["planning_area"] = residential_data["planning_area"].ffill()

    return residential_data

In [6]:
# resident_df = get_resident_data(2010)
# resident_df

In [7]:
def check_for_missing_subzone_pln_area(subzone_df, demographic_df):
    # check if the planning area and subzone name of both datasets match

    # 1. Perform the left join
    # subzones_2019 is the 'left' table because it contains all pairs
    check_df = pd.merge(
        subzone_df, 
        demographic_df, 
        left_on=['subzone_n', 'pln_area_n'], 
        right_on=['subzone', 'planning_area'], 
        how='left',
        indicator=True
    )

    # 2. Identify the pairs that are MISSING in residential_df
    missing_in_res = check_df[check_df['_merge'] == 'left_only']

    # 3. Identify the pairs that MATCH in both
    matching_pairs = check_df[check_df['_merge'] == 'both']

    print(f"Total pairs in Subzones df: {len(subzone_df)}")
    print(f"Matches found: {len(matching_pairs)}")
    print(f"Pairs missing in Residential df: {len(missing_in_res)}")
    if len(missing_in_res) != 0:
        print(f"Pairs missing in Residential df: {missing_in_res}")

In [8]:
# check_for_missing_subzone_pln_area(subzone_df, resident_df)

Density per hectare = $\frac{residential\ number \times 10000}{Area\ in \ m^2}$

In [9]:
def calculate_density(subzone_df, resident_df):
    # calculate the residential density of each subzone

    # left join to keep all subzones from the 2019 master list
    merged_total_density_df = pd.merge(
        subzone_df, 
        resident_df[['subzone', 'planning_area', 'total', 'total_above_60']], 
        left_on=['subzone_n', 'pln_area_n'], 
        right_on=['subzone', 'planning_area'], 
        how='left'
    )
    # We fill NaN totals with 0 so the calculation doesn't fail
    merged_total_density_df['total'] = merged_total_density_df['total'].fillna(0)
    merged_total_density_df['total_above_60'] = merged_total_density_df['total_above_60'].fillna(0)

    # Identify and sum all columns that contain 'residential' in the name
    residential_cols = [col for col in merged_total_density_df.columns if 'residential' in col]
    merged_total_density_df['total_residential_area'] = merged_total_density_df[residential_cols].sum(axis=1)

    # Calculate Density per Hectare
    # Formula: (residential number * 10000) / Area in m2
    merged_total_density_df['density_per_ha'] = (merged_total_density_df['total'] * 10000) / merged_total_density_df['total_residential_area']

    # Replace infinity values if area was 0
    merged_total_density_df['density_per_ha'] = merged_total_density_df['density_per_ha'].replace([np.inf, -np.inf], 0)
    # display(merged_total_density_df[['pln_area_n', 'subzone_n', 'total', 'total_residential_area', 'density_per_ha']].head())
    
    return merged_total_density_df

In [10]:
# merged_total_density_df = calculate_density(subzone_df, resident_df)

In [11]:
# def display_distribution(df, characteristic: str):
#     # Set the style
#     sns.set_theme(style="whitegrid")

#     # Create a figure with two subplots (1 row, 2 columns - or stacked)
#     # We will use a "Gridspec" style approach for a top-bottom view
#     f, (ax_box, ax_hist) = plt.subplots(2, sharex=True, gridspec_kw={"height_ratios": (.15, .85)}, figsize=(10, 7))

#     # 1. Plot the Range (Boxplot) on the top axis
#     sns.boxplot(x=df["density_per_ha"], ax=ax_box, color="lightseagreen")
#     ax_box.set(xlabel='') # Hide xlabel for the top plot
#     ax_box.set_title(f'Range and Distribution of {characteristic}', fontsize=15)

#     # 2. Plot the Distribution (Histogram + KDE) on the bottom axis
#     sns.histplot(df["density_per_ha"], ax=ax_hist, kde=True, color="teal", bins=20)

#     # Labeling
#     plt.xlabel(f'{characteristic}', fontsize=12)
#     plt.ylabel('Frequency (Subzone Count)', fontsize=12)

#     # Add a vertical line for the Median on the histogram
#     median_val = df["density_per_ha"].median()
#     ax_hist.axvline(median_val, color='red', linestyle='--', label=f'Median: {median_val:,.0f}')
#     plt.legend()

#     plt.tight_layout()
#     plt.show()

# display_distribution(merged_total_density_df, "Density per Hectare")

In [12]:
# merged_total_density_df["density_per_ha"] = merged_total_density_df["density_per_ha"].fillna(0)
# df = merged_total_density_df.sort_values("density_per_ha", ascending = False)
# df[['pln_area_n', 'subzone_n', 'total', 'total_residential_area', 'density_per_ha']].head(10)

Encoding resident density: 
- 0 residents -> 0
- 0 to 10 residents -> 1
- 10 to 50 residents -> 2
- 50 to 100 residents -> 3
- 100 to 1000 residents -> 4

There are 3 points of outlier that needs to be checked.
- people's park (0.09135 $km^2$)
- national university of s'pore (1.753 $km^2$)
- little india (0.2783 $km^2$)

landsizes base on: https://www.citypopulation.de/en/singapore/admin/

There are multiple subzones with residents but no recorded residential areas. They will be categorised as such:
1. subzones with 10 or less residents: 0 to 10 residents
2. subzones with 10 to 50 residents: 10 to 50 residents
3. subzones with 50 or more residents: 50 to 100 residents

Subzones with more than 100 residents but with no recorded residential areas will be assigned to the 50 to 100 residents category. These subzones have less established residential areas so its characteristics might differ from subzones with larger residential areas. This is also to avoid affecting the accuracy of prediction for subzones with more than 1000 residents.

People's park and national university of s'pore will be assigned to the 50 to 100 residents category. This is because both has a resident size of <300, People's park residents are mostly located within the people's park complex and national univeristy of singapore's residents would be students and staff living in accomodations throughout the university. 

little india will be assigned to the 100 to 1000 residents category as the subzone has a small area but more than 3000 residents

For *2015* subzone data, resident density for central water catchment will be subzones with 10 to 50 residents. Based on the total resident numbers for the subzone. 

In [13]:
def encode_resident_density(bins, labels, df, manual_overrides):
    # encode resident density base on the given bins and labels
    df["resident_density_encoding"] = pd.cut(df["density_per_ha"], bins = bins, labels = labels).astype(int)

    # Refined Logic for subzones with 0 density but >0 total residents
    cond1 = (df["resident_density_encoding"] == 0) & (df["total"] > 0) & (df["total"] <= 10)
    cond2 = (df["resident_density_encoding"] == 0) & (df["total"] > 10) & (df["total"] <= 50)
    cond3 = (df["resident_density_encoding"] == 0) & (df["total"] > 50)

    # Define the values to assign for each condition
    choices = [1, 2, 3]

    # Apply the correction
    df["resident_density_encoding"] = np.select(
        [cond1, cond2, cond3], 
        choices, 
        default=df["resident_density_encoding"]
    )

    # apply the manual mapping
    for subzone, new_encoding in manual_overrides.items():
        condition = df['subzone'].str.lower() == subzone.lower()
        df.loc[condition, "resident_density_encoding"] = new_encoding

    return df

In [14]:
# """
# 0 residents -> 0
# 0 to 10 residents -> 1
# 10 to 50 residents -> 2
# 50 to 100 residents -> 3
# 100 to 1000 residents -> 4
# """
# bins = [-0.1, 0, 10, 50, 100, float('inf')]
# labels = [0, 1, 2, 3, 4]

# # key is the subzone name, value is the density encoding you want to override with
# density_manual_overrides = {
#     "people's park": 3,
#     "national university of s'pore": 3,
#     "central water catchment": 2,
# }

# resident_density_df = encode_resident_density(bins, labels, merged_total_density_df, density_manual_overrides)

In [15]:
# resident_density_df.head()

#### Proportion of Residents aged >60 per 1000 residents
proportion of above age 60 residents = $\frac{residents\ above\ 60}{total\ subzone\ residents} \times 1000$

In [16]:
def encode_above_60_proportion(df):
    df['above_60_proportion'] = (df['total_above_60'] / df['total']) * 1000

    # Replace infinity values if area was 0
    df['above_60_proportion'] = df['above_60_proportion'].replace([np.inf, -np.inf], 0)
    df["above_60_proportion"] = df["above_60_proportion"].fillna(0)

    return df


In [17]:
# resident_above_60_df = encode_above_60_proportion(merged_total_density_df)
# resident_above_60_df

#### Ethnicity
Proportion of each ethnicity (chinses, malay, indian, others) per 1000 residents

In [18]:
def get_ethnicity_data(year: int):
    year = str(year)

    ethnicity_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/ethnicity_combined.xlsx")

    ethnicity_df = pd.read_excel(ethnicity_filepath, sheet_name = f"{year}")

    return ethnicity_df

In [19]:
# ethnicity_df = get_ethnicity_data(2015)
# ethnicity_df["planning_area"] = ethnicity_df["planning_area"].ffill()
# ethnicity_df.head()

In [20]:
# # check if there is any missing subzones for the ethnicity dataframe
# check_for_missing_subzone_pln_area(subzone_df, ethnicity_df)

In [21]:
def encode_ethnicity(subzone_df, ethnicity_df, columns_of_interest):

    columns_of_interest += ["subzone", "planning_area"]

    # left join to keep all subzones from the 2019 master list
    merged_ethnicity_density_df = pd.merge(
        subzone_df, 
        ethnicity_df[columns_of_interest], 
        left_on=['subzone_n', 'pln_area_n'],
        right_on=['subzone', 'planning_area'],
        how='left'
    )
    # We fill NaN totals with 0 so the calculation doesn't fail
    merged_ethnicity_density_df[columns_of_interest] = merged_ethnicity_density_df[columns_of_interest].fillna(0)

    # Calculate proportion of ethnicity per 1000 residents
    merged_ethnicity_density_df['male_chinese_proportion'] = (merged_ethnicity_density_df['male_chinese'] / merged_ethnicity_density_df['total']) * 1000
    merged_ethnicity_density_df['female_chinese_proportion'] = (merged_ethnicity_density_df['female_chinese'] / merged_ethnicity_density_df['total']) * 1000
    merged_ethnicity_density_df['male_malays_proportion'] = (merged_ethnicity_density_df['male_malays'] / merged_ethnicity_density_df['total']) * 1000
    merged_ethnicity_density_df['female_malays_proportion'] = (merged_ethnicity_density_df['female_malays'] / merged_ethnicity_density_df['total']) * 1000
    merged_ethnicity_density_df['male_indians_proportion'] = (merged_ethnicity_density_df['male_indians']/ merged_ethnicity_density_df['total']) * 1000
    merged_ethnicity_density_df['female_indians_proportion'] = (merged_ethnicity_density_df['female_indians']/ merged_ethnicity_density_df['total']) * 1000
    merged_ethnicity_density_df['male_others_proportion'] = (merged_ethnicity_density_df['male_others']/ merged_ethnicity_density_df['total']) * 1000
    merged_ethnicity_density_df['female_others_proportion'] = (merged_ethnicity_density_df['female_others']/ merged_ethnicity_density_df['total']) * 1000

    return merged_ethnicity_density_df

In [22]:
# columns_of_interest = ["total", "male_chinese", "female_chinese", "male_malays",
#                        "female_malays","male_indians", "female_indians",
#                        "male_others", "female_others"]
# merged_ethnicity_density_df = encode_ethnicity(subzone_df, ethnicity_df, columns_of_interest)
# merged_ethnicity_density_df

#### Workplace classification

In [23]:
def get_landuse_dataset(year: int):
    year = str(year)

    landuse_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    landuse_df = pd.read_csv(landuse_filepath)

    return landuse_df

In [24]:
# landuse_df = get_landuse_dataset(2014)
# landuse_df

There are 3 types of workplace classification in singapore.
- business 1: Clean industry, light industry, public utilities (logistics, warehousing, technology, research)
- business 2: Special industries involving industrial machinery, shipbuilding and repairing (heavy industries)
- business park: blend of industrial and office spaces

Each of these 3 workplaces could also be classified as white sites:

Which are areas used for commercial, hotel, residential, office, recreational club. As the number of workplaces that are labelled as white sites are few, they will be combined with their respective workplace types (eg: business 1 will combine with business 1 white)

Additionally, subzones with workplace areas smaller than 1000 $m^2$ will not be included. Reason is that a workplace that is too small would not have many workers in there.

references: 
- https://propertyreviewsg.com/ura-masterplan-zoning-interpretation/
- https://www.sgindustrialgroup.com/post/the-difference-between-different-industrial-property-classes

In [53]:
def determine_workplace_type(landuse_df, columns_of_interest):

    workplace_cols = [col for col in landuse_df.columns if 'business' in col]
    columns_of_interest += workplace_cols
    workplace_df = landuse_df[columns_of_interest].copy()

    workplace_df['pln_area_n'] = workplace_df['pln_area_n'].astype(str).str.strip().str.lower()
    workplace_df['subzone_n'] = workplace_df['subzone_n'].astype(str).str.strip().str.lower()

    workplace_df[workplace_cols] = workplace_df[workplace_cols].fillna(0)

    workplace_df['pri_classification'] = workplace_df['pri_classification'].astype(str).str.strip().str.lower()

    # group and Encode (Sum of workplace areas must be > 1000, else encode as 0)
    # sum the "White" and "Non-White" variants for each category
    workplace_df["business_1_encoding"] = (
        (workplace_df['business_1'] + workplace_df['business_1_-_white']) > 1000
    ).astype(int)
    workplace_df["business_2_encoding"] = (
        (workplace_df['business_2'] + workplace_df['business_2_-_white']) > 1000
    ).astype(int)
    workplace_df["business_park_encoding"] = (
        (workplace_df['business_park'] + workplace_df['business_park_-_white']) > 1000
    ).astype(int)

    return workplace_df

In [26]:
def determine_school_airport(landuse_df, columns_of_interest):

    school_cols = [col for col in landuse_df.columns if 'education' in col]
    columns_of_interest += school_cols
    df = landuse_df[columns_of_interest].copy()

    df['pln_area_n'] = df['pln_area_n'].astype(str).str.strip().str.lower()
    df['subzone_n'] = df['subzone_n'].astype(str).str.strip().str.lower()

    df[school_cols] = df[school_cols].fillna(0)

    # group and Encode (Sum of school areas must be > 1000, else encode as 0)
    # sum the "White" and "Non-White" variants for each category
    df["school_encoding"] = (
        df['educational_institution'] > 1000
    ).astype(int)

    airport_subzones = ['seletar aerospace park', 'changi airport']

    # Set 'airport' to 1 if subzone is in the list, else 0
    df['airport'] = df['subzone_n'].isin(airport_subzones).astype(int)

    return df

In [27]:
# columns_of_interest = ["subzone_n", "pln_area_n", "pri_classification"]

# workplace_df = determine_workplace_type(landuse_df, columns_of_interest)
# workplace_df.head()

#### Proportion of lower education level residents per 1000
In the Prediction model for future OHCAs based on geospatial and demogrpahic data paper, lower educational level was defined as "highest completed educational level being grade school, high school, vocational training or unknown educational level"

Lower education in Singapore's context would be post secondary (non-tertiary) education and below. 
#### Data.gov only has highest qualification attained by planning area, no subzone information could be found

In [28]:
# def get_lower_education_dataset(year: int):
#     year = str(year)
#     education = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/education_combined.xlsx")
#     education_df = pd.read_excel(education, sheet_name = year)

#     return education_df

In [29]:
# education_df = get_lower_education_dataset(2015)
# education_df

In [30]:
# print(len(subzone_df["pln_area_n"].unique()))

As the qualification dataset does not have information on all subzones in Singapore, subzones that are not listed in the highest qualification dataset will be assigned with 0 for lower education proportion.

In [31]:
# def lower_education_proportion(subzone_df, education_df, columns_of_interest):

#     columns_to_merge = ["planning_area", "total"] + columns_of_interest

#     # left join to keep all subzones from the 2019 master list
#     merged_education_proportion_df = pd.merge(
#         subzone_df, 
#         education_df[columns_to_merge], 
#         left_on=['pln_area_n'],
#         right_on=['planning_area'],
#         how='left'
#     )
#     # fill NaN totals with 0 so the calculation doesn't fail
#     merged_education_proportion_df[columns_of_interest] = merged_education_proportion_df[columns_of_interest].fillna(0)

#     # Identify and sum all columns that contain 'residential' in the name
#     merged_education_proportion_df['total_lower_education'] = merged_education_proportion_df[columns_of_interest].sum(axis=1)

#     # Calculate proportion of ethnicity per 1000 residents
#     # Using np.where to handle division by zero safely
#     merged_education_proportion_df['lower_education_proportion'] = np.where(
#         merged_education_proportion_df['total'] > 0,
#         (merged_education_proportion_df['total_lower_education'] / merged_education_proportion_df['total']) * 1000,
#         0
#     )

#     merged_education_proportion_df = merged_education_proportion_df[['pln_area_n', "total", "lower_education_proportion"]].copy()
#     merged_education_proportion_df = merged_education_proportion_df.fillna(0)
#     merged_education_proportion_df.drop_duplicates(inplace = True)

#     return merged_education_proportion_df

In [32]:
# columns_of_interest = ['no_qualification', 'pri', 'lower_sec', 'sec', 'post_sec_(non-tertiary)']

# education_proportion_df = lower_education_proportion(subzone_df, education_df, columns_of_interest)

# education_proportion_df.head()

#### Combining all the characteristics tgt

Combine from each dataframe:
- subzone: subzone_n, pln_area_n
- resident density: total, resident_density_encoding
- above 60: above_60_proportion
- lower education: lower_education_proportion
- workplace: business_1_encoding, business_2_encoding, business_park_encoding
- ethnicity: chinese_proportion, malays_proportion, indians_proportion, others_proportion

In [58]:
# def combine_subzone_characteristics(merged_total_density_df, merged_ethnicity_density_df, workplace_df, education_proportion_df):
def combine_subzone_characteristics(merged_total_density_df, merged_ethnicity_density_df, workplace_df, school_df):

    subzone_combined_characteristics = merged_total_density_df[["pln_area_n", "subzone_n", "total", "resident_density_encoding", "above_60_proportion", "year"]].copy()

    # get ethnicity proportion
    subzone_combined_characteristics = pd.merge(
        subzone_combined_characteristics,
        merged_ethnicity_density_df[["pln_area_n", "subzone_n",
                                    'male_chinese_proportion',
                                    'female_chinese_proportion',
                                    'male_malays_proportion',
                                    'female_malays_proportion',
                                    'male_indians_proportion',
                                    'female_indians_proportion',
                                    'male_others_proportion',
                                    'female_others_proportion',
                                    'year']],
        on = ["pln_area_n", "subzone_n"],
        how = "left"
    )

    # get workplace classification
    subzone_combined_characteristics = pd.merge(
        subzone_combined_characteristics,
        workplace_df[["pln_area_n", "subzone_n", "pri_classification",
                                    "business_1_encoding", "business_2_encoding",
                                    "business_park_encoding", 'year']],
        on = ["pln_area_n", "subzone_n"],
        how = "left"
    )

    subzone_combined_characteristics = pd.merge(
        subzone_combined_characteristics,
        school_df[["pln_area_n", "subzone_n",
                                    "school_encoding", "airport"]],
        on = ["pln_area_n", "subzone_n"],
        how = "left"
    )

    # subzone_combined_characteristics = pd.merge(
    #     subzone_combined_characteristics,
    #     education_proportion_df[["pln_area_n", "lower_education_proportion"]],
    #     on = ["pln_area_n"],
    #     how = "left"
    # )

    return subzone_combined_characteristics

In [34]:
# subzone_combined_characteristics = combine_subzone_characteristics(merged_total_density_df, merged_ethnicity_density_df, workplace_df, education_proportion_df)
# subzone_combined_characteristics

In [35]:
# import pandas as pd

# def check_for_missing_subzone_pln_area(subzone_df, demographic_df):
#     # 1. Perform an outer join to capture mismatches from both sides
#     check_df = pd.merge(
#         subzone_df, 
#         demographic_df, 
#         left_on=['subzone_n', 'pln_area_n'], 
#         right_on=['subzone', 'planning_area'], 
#         how='outer',
#         indicator=True
#     )

#     # 2. Pairs in subzone_df but missing in demographic_df
#     missing_in_demographic = check_df[check_df['_merge'] == 'left_only']

#     # 3. Pairs in demographic_df but missing in subzone_df (What you asked for)
#     missing_in_subzone = check_df[check_df['_merge'] == 'right_only']

#     # 4. Pairs that match in both
#     matching_pairs = check_df[check_df['_merge'] == 'both']

#     print(f"Total pairs in Subzones df: {len(subzone_df)}")
#     print(f"Total pairs in Demographic df: {len(demographic_df)}")
#     print("-" * 30)
#     print(f"Matches found: {len(matching_pairs)}")
#     print(f"Pairs in Subzone df but NOT in Demographic df: {len(missing_in_demographic)}")
#     print(f"Pairs in Demographic df but NOT in Subzone df: {len(missing_in_subzone)}")

#     if not missing_in_subzone.empty:
#         print("\nPairs existing in Demographic df but missing in Subzone df:")
#         # Selecting only the demographic columns for clarity
#         print(missing_in_subzone[['subzone', 'planning_area']])

#     return missing_in_subzone # Optional: return the dataframe for further cleaning

# missing_in_subzone = check_for_missing_subzone_pln_area(subzone_df, resident_df)
# # save_to_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/missing_pairs_from_subzone_2010.csv")
# # missing_in_subzone.to_csv(save_to_filepath)

By visually checking which subzone in 2008 overlaps with 2014 subzones, I have mapped the 2014 subzone names to the 2008 ones. 

In [ ]:
mapping_data = [
    ("jurong east", "boon lay", "yuhua west"),
    ("jurong west", "boon lay", "boon lay place"),
    ("sengkang", "buangkok", "anchorvale"),
    ("bukit panjang", "bukit panjang", "jelebu"),
    ("choa chu kang", "central", "choa chu kang central"),
    ("jurong west", "central", "jurong west central"),
    ("pasir ris", "elias", "pasir ris west"),
    ("bukit batok", "hong kah", "hong kah north"),
    ("sengkang", "jalan kayu east", "fernvale"),
    ("sengkang", "jalan kayu west", "sengkang west"),
    ("jurong east", "jurong lake", "lakeside"),
    ("jurong east", "jurong regional centre", "jurong gateway"),
    ("toa payoh", "kallang", "lorong 8 toa payoh"),
    ("choa chu kang", "kranji north", "choa chu kang north"),
    ("toa payoh", "marymount", "toa payoh west"),
    ("choa chu kang", "pang sua", "choa chu kang north"),
    ("pasir ris", "pasir ris town", "pasir ris central"),
    ("toa payoh", "paya lebar", "joo seng"),
    ("hougang", "rosyth", "kovan"),
    ("ang mo kio", "sindo", "tagore"),
    ("punggol", "subzone 2", "punggol town centre"),
    ("punggol", "subzone 3", "matilda"),
    ("punggol", "subzone 4", "punggol field"),
    ("punggol", "subzone 5", "waterway east"),
    ("sengkang", "sungei serangoon west", "rivervale"),
    ("hougang", "tai keng", "tai seng"),
    ("ang mo kio", "town centre", "ang mo kio town centre"),
    ("sengkang", "trafalgar", "compassvale"),
    ("jurong east", "yuhua", "yuhua east")
]

In [37]:
def get_subzone_densities(mapping_data):
    demographic_subzone_mapping = {2010: 2008,
                                   2015: 2014,
                                   2020: 2019}
    output = []
    for demographic_year, subzone_year in demographic_subzone_mapping.items():
        current_subzone_year = subzone_year
        resident_df = get_resident_data(demographic_year)
        # there is no landuse data for the 2008 masterplan, 
        # so I will be using the landuse data from 2014
        if subzone_year == 2008:
            current_subzone_year = 2014
            
            map_df = pd.DataFrame(mapping_data, columns=['planning_area', 'subzone', 'new_subzone'])
            # renaming certain subzone names so they would match with subzone names from 2014
            # Merge this with your resident_df to update the names
            resident_df = resident_df.merge(
                map_df, 
                on=['planning_area', 'subzone'], 
                how='left'
            )

            # Replace the old subzone with the new one where a match was found
            resident_df['subzone'] = resident_df['new_subzone'].fillna(resident_df['subzone'])
            resident_df.drop(columns=['new_subzone'], inplace=True)

        subzone_df = get_subzone_data(current_subzone_year)
        print(f"checking for subzones for the year {subzone_year} for demographic data from {demographic_year}")
        check_for_missing_subzone_pln_area(subzone_df, resident_df)

        merged_total_density_df = calculate_density(subzone_df, resident_df)
        # add year column for easier identification of rows
        merged_total_density_df['year'] = demographic_year

        merged_total_density_df["density_per_ha"] = merged_total_density_df["density_per_ha"].fillna(0)
        df = merged_total_density_df.sort_values("density_per_ha", ascending = False)
        print("Displaying 10 most dense subzones by resident count:")
        display(df[['pln_area_n', 'subzone_n', 'total', 'total_residential_area', 'density_per_ha']].head(10))
        
        output.append(merged_total_density_df)
    
    return output
        

In [38]:
# The table outputs should only contain subzones with the name total
# The 3 years of dataset should have 191, 323, and 332 rows from 2010 to 2020 respectively
subzone_resident_densities = get_subzone_densities(mapping_data)

checking for subzones for the year 2008 for demographic data from 2010
Total pairs in Subzones df: 323
Matches found: 191
Pairs missing in Residential df: 133
Pairs missing in Residential df:                      subzone_n               pln_area_n  beach_area  \
9            yio chu kang east               ang mo kio         0.0   
10          yio chu kang north               ang mo kio         0.0   
11           yio chu kang west               ang mo kio         0.0   
23                    liu fang                 boon lay         0.0   
24                     samulun                 boon lay         0.0   
..                         ...                      ...         ...   
305    western water catchment  western water catchment         0.0   
306             greenwood park                woodlands         0.0   
309                senoko west                woodlands         0.0   
312  woodlands regional centre                woodlands         0.0   
317                   nee s

,pln_area_n,subzone_n,total,total_residential_area,density_per_ha
8,ang mo kio,yio chu kang,31443.0,33365.421743,9423.828130
107,hougang,hougang central,50316.0,265180.362446,1897.425569
225,rochor,little india,4164.0,31678.627708,1314.450878
115,hougang,trafalgar,95922.0,884092.198605,1084.977338
173,outram,china square,1788.0,20798.118907,859.693133
254,serangoon,serangoon central,53506.0,626128.079739,854.553593
137,kallang,crawford,10032.0,121057.455377,828.697412
133,jurong west,wenya,8190.0,101520.563860,806.733108
174,outram,chinatown,9253.0,121098.901133,764.086207
129,jurong west,jurong west central,69324.0,935811.205248,740.790446


checking for subzones for the year 2014 for demographic data from 2015
Total pairs in Subzones df: 323
Matches found: 323
Pairs missing in Residential df: 0
Displaying 10 most dense subzones by resident count:


,pln_area_n,subzone_n,total,total_residential_area,density_per_ha
175,outram,people's park,390,0.001253,3.113257e+09
208,queenstown,national university of s'pore,360,0.010230,3.518961e+08
68,central water catchment,central water catchment,10,0.184207,5.428676e+05
224,rochor,little india,3850,31678.627708,1.215330e+03
173,outram,chinatown,11880,121098.901133,9.810163e+02
132,jurong west,wenya,8610,101520.563860,8.481040e+02
136,kallang,crawford,9740,121057.455377,8.045766e+02
172,outram,china square,1590,20798.118907,7.644922e+02
71,changi,changi west,1740,23721.594870,7.335089e+02
128,jurong west,jurong west central,68200,935811.205248,7.287795e+02


checking for subzones for the year 2019 for demographic data from 2020
Total pairs in Subzones df: 332
Matches found: 332
Pairs missing in Residential df: 0
Displaying 10 most dense subzones by resident count:


,pln_area_n,subzone_n,total,total_residential_area,density_per_ha
177,outram,people's park,290,0.001252,2.316001e+09
210,queenstown,national university of s'pore,240,0.003253,7.378667e+08
226,rochor,little india,3270,31642.338720,1.033426e+03
175,outram,chinatown,10490,115219.825786,9.104336e+02
134,jurong west,wenya,8280,101519.967978,8.156031e+02
58,bukit panjang,saujana,25920,359889.326428,7.202214e+02
238,sembawang,sembawang central,34800,488491.882146,7.123967e+02
76,choa chu kang,peng siang,34720,496286.705139,6.995956e+02
138,kallang,crawford,8380,121057.691564,6.922319e+02
130,jurong west,jurong west central,63760,926456.931959,6.882133e+02


Encoding resident density: 
- 0 residents -> 0
- 0 to 10 residents -> 1
- 10 to 50 residents -> 2
- 50 to 100 residents -> 3
- 100 to 1000 residents -> 4

There are 3 points of outlier that needs to be checked.
- people's park (0.09135 $km^2$)
- national university of s'pore (1.753 $km^2$)
- little india (0.2783 $km^2$)

landsizes base on: https://www.citypopulation.de/en/singapore/admin/

There are multiple subzones with residents but no recorded residential areas. They will be categorised as such:
1. subzones with 10 or less residents: 0 to 10 residents
2. subzones with 10 to 50 residents: 10 to 50 residents
3. subzones with 50 or more residents: 50 to 100 residents

Subzones with more than 100 residents but with no recorded residential areas will be assigned to the 50 to 100 residents category. These subzones have less established residential areas so its characteristics might differ from subzones with larger residential areas. This is also to avoid affecting the accuracy of prediction for subzones with more than 1000 residents.

People's park and national university of s'pore will be assigned to the 50 to 100 residents category. This is because both has a resident size of <300, People's park residents are mostly located within the people's park complex and national univeristy of singapore's residents would be students and staff living in accomodations throughout the university. 

little india will be assigned to the 100 to 1000 residents category as the subzone has a small area but more than 3000 residents

For *2015* subzone data, resident density for central water catchment will be subzones with 10 to 50 residents. Based on the total resident numbers for the subzone. 

In [39]:
# key is the subzone name, value is the density encoding you want to override with
density_manual_overrides = {
        "people's park": 3,
        "national university of s'pore": 3,
        "central water catchment": 2,
    }

In [40]:
def encode_densities(subzone_resident_densities, density_manual_overrides):

    bins = [-0.1, 0, 10, 50, 100, float('inf')]
    labels = [0, 1, 2, 3, 4]

    output = []
    for resident_densities_df in subzone_resident_densities:
        resident_density_df = encode_resident_density(bins, labels, resident_densities_df, density_manual_overrides)
        output.append(resident_density_df)
    return output

In [41]:
encoded_density_df = encode_densities(subzone_resident_densities, density_manual_overrides)


#### Proportion of Residents aged >60 per 1000 residents

In [42]:
def encode_seniors(subzone_resident_densities):
    years = [2008, 2014, 2019]
    output = []
    for i, df in enumerate(subzone_resident_densities):
        resident_above_60_df = encode_above_60_proportion(df)
        # resident_above_60_df['year'] = years[i]
        output.append(resident_above_60_df)
    return output

In [43]:
encoded_density_seniors_df = encode_seniors(subzone_resident_densities)
# encoded_density_seniors_df[1]

#### Ethnicity Proportion

In [44]:
def ethnicity_proportion(df, mapping_data):
    demographic_subzone_mapping = {2010: 2008,
                                   2015: 2014,
                                   2020: 2019}
    output = []

    for i, (demographic_year, subzone_year) in enumerate(demographic_subzone_mapping.items()):
        ethnicity_df = get_ethnicity_data(demographic_year)
        ethnicity_df["planning_area"] = ethnicity_df["planning_area"].ffill()
        current_subzone_year = subzone_year

        combined_encoded_df = df[i]

        # there is no landuse data for the 2008 masterplan, 
        # so I will be using the landuse data from 2014
        if subzone_year == 2008:
            current_subzone_year = 2014
            
            map_df = pd.DataFrame(mapping_data, columns=['planning_area', 'subzone', 'new_subzone'])
            # renaming certain subzone names so they would match with subzone names from 2014
            # Merge this with your ethnicity_df to update the names
            ethnicity_df = ethnicity_df.merge(
                map_df, 
                on=['planning_area', 'subzone'], 
                how='left'
            )

            # Replace the old subzone with the new one where a match was found
            ethnicity_df['subzone'] = ethnicity_df['new_subzone'].fillna(ethnicity_df['subzone'])
            ethnicity_df.drop(columns=['new_subzone'], inplace=True)

        subzone_df = get_subzone_data(current_subzone_year)
        check_for_missing_subzone_pln_area(subzone_df, ethnicity_df)

        columns_of_interest = ["total", "male_chinese", "female_chinese", "male_malays",
                            "female_malays","male_indians", "female_indians",
                            "male_others", "female_others"]
        combined_encoded_df = encode_ethnicity(subzone_df, ethnicity_df, columns_of_interest)
        combined_encoded_df['year'] = demographic_year
        output.append(combined_encoded_df)

    return output

In [45]:
encoded_ethnicity_df = ethnicity_proportion(encoded_density_seniors_df, mapping_data)


Total pairs in Subzones df: 323
Matches found: 191
Pairs missing in Residential df: 133
Pairs missing in Residential df:                      subzone_n               pln_area_n  beach_area  \
9            yio chu kang east               ang mo kio         0.0   
10          yio chu kang north               ang mo kio         0.0   
11           yio chu kang west               ang mo kio         0.0   
23                    liu fang                 boon lay         0.0   
24                     samulun                 boon lay         0.0   
..                         ...                      ...         ...   
305    western water catchment  western water catchment         0.0   
306             greenwood park                woodlands         0.0   
309                senoko west                woodlands         0.0   
312  woodlands regional centre                woodlands         0.0   
317                   nee soon                   yishun         0.0   

       business_2  educati

In [54]:
def encode_workplace():
    demographic_subzone_mapping = {2010: 2008,
                                   2015: 2014,
                                   2020: 2019}
    
    output = []
    for i, (demographic_year, subzone_year) in enumerate(demographic_subzone_mapping.items()):
        current_subzone_year = subzone_year
        # 2008 workplace data will be from 2014 masterplan
        if subzone_year == 2008:
            current_subzone_year = 2014
        landuse_df = get_landuse_dataset(current_subzone_year)

        columns_of_interest = ["subzone_n", "pln_area_n", "pri_classification"]

        workplace_df = determine_workplace_type(landuse_df, columns_of_interest)
        workplace_df['year'] = demographic_year
        output.append(workplace_df)

    return output

In [55]:
encoded_workplace = encode_workplace()
encoded_workplace[1]

,subzone_n,pln_area_n,pri_classification,business_2,business_1_-_white,business_1,business_2_-_white,business_park_-_white,business_park,business_1_encoding,business_2_encoding,business_park_encoding,year
0,ang mo kio town centre,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
1,cheng san,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
2,chong boon,ang mo kio,residential,0.0,0.0,161265.128547,0.0,0.0,0.0,1,0,0,2015
3,kebun bahru,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
4,sembawang hills,ang mo kio,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...
318,springleaf,yishun,reserve site,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
319,yishun central,yishun,reserve site,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
320,yishun east,yishun,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015
321,yishun south,yishun,residential,0.0,0.0,0.000000,0.0,0.0,0.0,0,0,0,2015


In [48]:
def encode_schools_airport():
    demographic_subzone_mapping = {2010: 2008,
                                   2015: 2014,
                                   2020: 2019}
    
    output = []
    for i, (demographic_year, subzone_year) in enumerate(demographic_subzone_mapping.items()):
        current_subzone_year = subzone_year
        # 2008 workplace data will be from 2014 masterplan
        if subzone_year == 2008:
            current_subzone_year = 2014
        landuse_df = get_landuse_dataset(current_subzone_year)

        columns_of_interest = ["subzone_n", "pln_area_n", "pri_classification"]

        school_airport_df = determine_school_airport(landuse_df, columns_of_interest)
        school_airport_df['year'] = demographic_year
        output.append(school_airport_df)

    return output

In [50]:
encoded_school_airport = encode_schools_airport()
# encoded_school_airport[1]

In [59]:
demographic_subzone_mapping = {2010: 2008,
                                2015: 2014,
                                2020: 2019}

for i, (demographic_year, subzone_year) in enumerate(demographic_subzone_mapping.items()):
    subzone_combined_characteristics = combine_subzone_characteristics(encoded_density_seniors_df[i], encoded_ethnicity_df[i],
                                                                       encoded_workplace[i], encoded_school_airport[i])
    
    save_to_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_combined_characteristics_{demographic_year}.csv")
    subzone_combined_characteristics.to_csv(save_to_filepath)
